In [1]:
import yaml
import glob
import json
import os
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pprint import pprint
from typing import Dict, Optional, List, Any, Tuple
from slideguard.schemes import FullEvaluation
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.language_models import LanguageModelInput

# Functions

In [2]:
class VLLMChatOpenAI(ChatOpenAI):
    def _get_request_payload(
        self,
        input_: LanguageModelInput,
        *,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> dict:
        payload = super()._get_request_payload(input_, stop=stop, **kwargs)
        # max_tokens was deprecated in favor of max_completion_tokens
        # in September 2024 release
        if "max_completion_tokens" in payload:
            payload["max_tokens"] = payload.pop("max_completion_tokens")
        return payload
    

def load_goldens(base_path: str = "../golden") -> Dict[str, dict]:
    golden_files = glob.glob(os.path.join(base_path, "*.yaml"))

    goldens = dict()
    for file in golden_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        with open(file, "r") as f:
            evaluation = yaml.safe_load(f)
        goldens[deck_name] = evaluation
    
    return goldens


def load_evaluations(base_path: str = "../slidedecks_test_evaluations") -> Dict[str, FullEvaluation]:
    evaluation_files = glob.glob(os.path.join(base_path, "evaluations_*.json"))

    evaluations = dict()
    for file in evaluation_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        deck_name = deck_name.replace("evaluations_", "")
        with open(file, "r") as f:
            evaluation = FullEvaluation.model_validate_json(f.read())
        evaluations[deck_name] = evaluation
    
    return evaluations

In [3]:
system_prompt = """
You are a helpful assistant.
You need to estimate if a golden comment made by a human expert is presented in the set of evaluation comments made by an AI agent. 
The expert's comment may have a different structure and form than the evaluation comment, 
but the essence of the comment should be the same.
AI agent may have many comments in the set, so you need to find at least one comment that is the most similar to the expert's comment.


Write your answer in the following JSON format:
{json_schema}
"""

human_prompt = """
The comment made by a human expert:
{human_comment}

The set of evaluations comment made by an AI agent:
{ai_comments}
"""

class JudgementAnswer(BaseModel):
    citation: List[str] = Field(description="The closest comment (one or more) from the set of evaluation comments to the expert's comment")
    judgement: bool = Field(description="Whether the expert's comment is present in the set of evaluation comments")


llm = VLLMChatOpenAI(
    model="/model",
    temperature=0.1,
    max_completion_tokens=1000,
    max_tokens=1000,
    base_url="http://d.dgx:8082/v1",
    api_key="token-abc123"
)

parser = PydanticOutputParser(pydantic_object=JudgementAnswer)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt)
])

chain = chat_prompt | llm | parser

In [33]:
goldens = load_goldens()
evaluations = load_evaluations()

print("Goldens:", len(goldens))
print("Evaluations:", len(evaluations))

Goldens: 3
Evaluations: 3


In [41]:
from typing import Dict, Optional
from slideguard.schemes import FullEvaluation
from pydantic import BaseModel
from tqdm import tqdm


class RecordOfMetrics(BaseModel):
    deck_name: str
    criteria: str
    expert_comment: str
    ai_comments: List[str]
    judgement: bool
    slide_id: Optional[int] = None

class RecordsBatch(BaseModel):
    records: List[RecordOfMetrics]

def deck_judge_evaluations(goldens: Dict[str, dict], evaluations: Dict[str, FullEvaluation], selected_deck_name: Optional[str] = None):  
    golden2evaluation: Dict[str, Tuple[dict, FullEvaluation]] = dict()
    for deck_name, golden in goldens.items():
        if deck_name in evaluations:
            golden2evaluation[deck_name] = (golden, evaluations[deck_name])
        else:
            print(f"No evaluation for {deck_name}")

    print(f"Loaded {len(golden2evaluation)} decks")

    if selected_deck_name is not None:
        if selected_deck_name not in golden2evaluation:
            raise ValueError(f"No data for {selected_deck_name}")
        decks = [selected_deck_name]
    else:
        decks = list(golden2evaluation.keys())

    computables = []
    for deck_name in decks:
        print(f"Preparing {deck_name}")
        golden, evaluation = golden2evaluation[deck_name]

        golden_slides_comments = dict()
        for el in golden['evaluation']['slides']:
            slide_id = el['slide_id']
            
            if slide_id not in golden_slides_comments:
                golden_slides_comments[slide_id] = dict()
            
            if el['criteria'] not in golden_slides_comments[slide_id]:
                golden_slides_comments[slide_id][el['criteria']] = []
            
            golden_slides_comments[slide_id][el['criteria']].append(el['comment'])

        # slide metrics
        slide_computables = []
        for slide_evaluation in evaluation.slide_evaluations:
            for criteria, evaluation_element in slide_evaluation.evaluations.items():
                eval_results = [el for el in evaluation_element['evaluation_results'] if el['severity'] > 2]

                if slide_evaluation.slide_id not in golden_slides_comments or criteria not in golden_slides_comments[slide_evaluation.slide_id]:
                    continue
                
                for comment in golden_slides_comments[slide_evaluation.slide_id][criteria]:
                    slide_computables.append({
                        "human_comment": comment,
                        "ai_comments": str(eval_results),
                        "json_schema": parser.get_format_instructions(),
                        "deck_name": deck_name,
                        "criteria": criteria,
                        "expert_comment": comment,
                        "slide_id": slide_evaluation.slide_id
                    })
        
        print(f"Found {len(slide_computables)} slides records for {deck_name}")
        
        # deck metrics
        deck_computables = []
        for criteria, evaluation in evaluation.deck_evaluations.evaluations.items():
            eval_results = [el for el in evaluation['evaluation_results'] if el['severity'] > 2]
        
            if criteria not in golden['evaluation']['deck']:
                continue
            
            for comment in golden['evaluation']['deck'][criteria]:
                deck_computables.append({
                    "human_comment": comment,
                    "ai_comments": str(eval_results),
                    "json_schema": parser.get_format_instructions(),
                    "deck_name": deck_name,
                    "criteria": criteria,
                    "expert_comment": comment,
                    "slide_id": None
                })
        print(f"Found {len(deck_computables)} deck records for {deck_name}")

        computables.extend(slide_computables)
        computables.extend(deck_computables)
    
    results: List[JudgementAnswer] = []
    for idx, result in tqdm(chain.batch_as_completed(computables), total=len(computables), desc="Judging"):
        results.append((idx, result))

    results = sorted(results, key=lambda x: x[0])
    results = [result for _, result in results]
    
    records = [
        RecordOfMetrics(
            deck_name=input_['deck_name'],
            criteria=input_['criteria'],
            expert_comment=input_['expert_comment'],
            ai_comments=result.citation,
            judgement=result.judgement,
            slide_id=input_['slide_id']
        )
        for input_, result in zip(computables, results)
    ]
                
    return records


In [38]:
records = deck_judge_evaluations(goldens, evaluations)

Loaded 3 decks
Preparing 1_EN_Kataeva_Thesis
Found 7 slides records for 1_EN_Kataeva_Thesis
Found 5 deck records for 1_EN_Kataeva_Thesis
Preparing 15_RU_Basilaev_Thesis
Found 14 slides records for 15_RU_Basilaev_Thesis
Found 11 deck records for 15_RU_Basilaev_Thesis
Preparing 12_EN_Zamiralov_NIR
Found 12 slides records for 12_EN_Zamiralov_NIR
Found 13 deck records for 12_EN_Zamiralov_NIR


Judging: 100%|██████████| 62/62 [00:10<00:00,  5.87it/s]


In [39]:
import itertools

for (deck_name, criteria), criteria_records in itertools.groupby(
        sorted(records, key=lambda x: (x.deck_name, x.criteria)), key=lambda x: (x.deck_name, x.criteria)
    ):
    criteria_records: List[RecordOfMetrics] = list(criteria_records)
    positive = sum(record.judgement for record in criteria_records)
    print(f"{deck_name} {criteria}: {positive} / {len(criteria_records)} = {(positive / len(criteria_records) * 100):.2f}%")

12_EN_Zamiralov_NIR deck_storytelling: 4 / 6 = 66.67%
12_EN_Zamiralov_NIR deck_structure_analysis: 6 / 7 = 85.71%
12_EN_Zamiralov_NIR slide_graphic_content_match: 0 / 5 = 0.00%
12_EN_Zamiralov_NIR slide_title_content_match: 1 / 1 = 100.00%
12_EN_Zamiralov_NIR slide_title_slide_quality: 1 / 2 = 50.00%
12_EN_Zamiralov_NIR slide_visual_arrangement: 1 / 4 = 25.00%
15_RU_Basilaev_Thesis deck_research_quality: 1 / 2 = 50.00%
15_RU_Basilaev_Thesis deck_storytelling: 5 / 9 = 55.56%
15_RU_Basilaev_Thesis slide_graphic_content_match: 0 / 4 = 0.00%
15_RU_Basilaev_Thesis slide_title_slide_quality: 2 / 3 = 66.67%
15_RU_Basilaev_Thesis slide_visual_arrangement: 3 / 7 = 42.86%
1_EN_Kataeva_Thesis deck_storytelling: 2 / 3 = 66.67%
1_EN_Kataeva_Thesis deck_structure_analysis: 0 / 2 = 0.00%
1_EN_Kataeva_Thesis slide_graphic_content_match: 0 / 1 = 0.00%
1_EN_Kataeva_Thesis slide_visual_arrangement: 2 / 6 = 33.33%


In [44]:
RecordsBatch(records=records)

ValidationError: 62 validation errors for RecordsBatch
records.0
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name...gement=True, slide_id=2), input_type=RecordOfMetrics]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
records.1
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name...ement=False, slide_id=4), input_type=RecordOfMetrics]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
records.2
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name...ement=False, slide_id=7), input_type=RecordOfMetrics]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
records.3
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name..

In [42]:
with open('metric_records.json', 'w') as f:
    f.write(RecordsBatch(records=records).model_dump_json(indent=4))


ValidationError: 62 validation errors for RecordsBatch
records.0
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name...gement=True, slide_id=2), input_type=RecordOfMetrics]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
records.1
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name...ement=False, slide_id=4), input_type=RecordOfMetrics]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
records.2
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name...ement=False, slide_id=7), input_type=RecordOfMetrics]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
records.3
  Input should be a valid dictionary or instance of RecordOfMetrics [type=model_type, input_value=RecordOfMetrics(deck_name..